# Levanti / Palestinian Harakat Prediction Maker

This notebook applies the off-the-shelf Hugging Face model `guymorlan/levanti_arabic2diacritics` to Arabic text without harakat. It is set up for Google Colab and mirrors the practical flow of `harakat_prediction_maker_script.ipynb`: load data, resume saved predictions, run the model over `arabic_stripped`, save progress, and calculate exact-match metrics against `arabic_harakat` when available.

Model page: https://huggingface.co/guymorlan/levanti_arabic2diacritics

In [8]:
####################################
# install dependencies for Colab
####################################
!pip install -q transformers accelerate

In [9]:
####################################
# imports and global configuration
####################################
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, RobertaForTokenClassification

MODEL_ID = "guymorlan/levanti_arabic2diacritics"
SOURCE_COL = "arabic_stripped"
TARGET_COL = "arabic_harakat"

# Smaller values save more often, which is safer for long Colab runs.
SAVE_EVERY_ROWS = 100
MAX_ROWS = None  # Set to a small integer for testing. Use None for the full file.

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


In [10]:
####################################
# file paths
####################################
# In Colab, this uses the arabizi folder at the top of MyDrive. Locally, it
# falls back to project folders so the notebook can still be tested.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ARABIZI_DIR = Path("/content/drive/MyDrive/arabizi")
    DATA_FILE = ARABIZI_DIR / "maknuune-v1.0.1_cleaned.csv"
except ModuleNotFoundError:
    PROJECT_DIR = Path.cwd()
    if PROJECT_DIR.name == "train_harakat_adder":
        PROJECT_DIR = PROJECT_DIR.parents[1]
    candidates = [
        PROJECT_DIR / "working_files" / "data" / "maknuune-v1.0.1_cleaned.csv",
        PROJECT_DIR / "maknuune-v1.0.1_cleaned.csv",
        PROJECT_DIR / "data" / "maknuune-v1.0.1_cleaned.csv",
    ]
    DATA_FILE = next((path for path in candidates if path.exists()), candidates[0])
    ARABIZI_DIR = DATA_FILE.parent

PREDICTIONS_FILE = ARABIZI_DIR / "maknuune-v1.0.1_cleaned_levanti_harakat_predictions.csv"
METRICS_FILE = ARABIZI_DIR / "levanti_harakat_prediction_metrics_by_length.csv"

print("Arabizi directory:", ARABIZI_DIR)
print("Data file:", DATA_FILE)
print("Predictions file:", PREDICTIONS_FILE)
print("Metrics file:", METRICS_FILE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Arabizi directory: /content/drive/MyDrive/arabizi
Data file: /content/drive/MyDrive/arabizi/maknuune-v1.0.1_cleaned.csv
Predictions file: /content/drive/MyDrive/arabizi/maknuune-v1.0.1_cleaned_levanti_harakat_predictions.csv
Metrics file: /content/drive/MyDrive/arabizi/levanti_harakat_prediction_metrics_by_length.csv


In [11]:
####################################
# load off-the-shelf Palestinian/Levanti diacritizer
####################################
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = RobertaForTokenClassification.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()

# Model card labels. The model predicts multiple possible marks per character,
# because shadda can combine with a vowel or sukun.
LABEL_TO_DIACRITIC = {
    0: "ّ",  # shadda
    1: "َ",  # fatha
    2: "ِ",  # kasra
    3: "ُ",  # damma
    4: "ْ",  # sukun
}

print("Loaded model:", MODEL_ID)
print("Model labels:", model.config.id2label)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: guymorlan/levanti_arabic2diacritics
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model: guymorlan/levanti_arabic2diacritics
Model labels: {0: 'LABEL_0', 1: 'LABEL_1', 2: 'LABEL_2', 3: 'LABEL_3', 4: 'LABEL_4'}


In [12]:
####################################
# text helpers and prediction function
####################################
def now_iso() -> str:
    return datetime.now().isoformat(timespec="seconds")


def clean_text(value: object) -> str:
    if pd.isna(value):
        return ""
    return " ".join(str(value).split())


@torch.no_grad()
def add_levanti_harakat(text: str) -> str:
    """Add Palestinian/Levanti-style harakat to raw Arabic text."""
    text = clean_text(text)
    if not text:
        return ""

    tokens = tokenizer(text, return_tensors="pt", truncation=True).to(DEVICE)
    logits = model(**tokens).logits
    predictions = (logits.sigmoid() > 0.5)[0]

    # The model card uses [1:-1] to drop BOS/EOS for this character-level model.
    char_predictions = predictions[1:-1]
    if len(char_predictions) != len(text):
        # Fallback keeps the notebook from silently misaligning text. This should
        # be rare for normal Arabic strings with this tokenizer.
        raise ValueError(
            f"Prediction/text length mismatch: {len(char_predictions)} predictions for {len(text)} characters. "
            "Try shortening the text or checking tokenizer alignment."
        )

    output = []
    for pred, char in zip(char_predictions, text):
        output.append(char)
        # Add vowel/sukun marks before shadda, matching the model-card example.
        for label_idx in range(1, 5):
            if bool(pred[label_idx]):
                output.append(LABEL_TO_DIACRITIC[label_idx])
        if bool(pred[0]):
            output.append(LABEL_TO_DIACRITIC[0])
    return "".join(output)


example = "بديش اروح عالمدرسة بكرا"
print("Input:", example)
print("Prediction:", add_levanti_harakat(example))

Input: بديش اروح عالمدرسة بكرا
Prediction: بِدِّيْش اْرُوْح عَالْمَدْرَسِة بُكْرَا


In [13]:
####################################
# load data and resume saved predictions
####################################
PREDICTION_COLUMNS = [
    "levanti_harakat_prediction",
    "levanti_harakat_prediction_done_at",
]

if not DATA_FILE.exists():
    raise FileNotFoundError(f"Could not find cleaned data file: {DATA_FILE}")

if PREDICTIONS_FILE.exists():
    df = pd.read_csv(PREDICTIONS_FILE)
    print("Resuming from existing Levanti predictions file.")
else:
    df = pd.read_csv(DATA_FILE)
    print("Starting from the cleaned data file.")

required_columns = {SOURCE_COL}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns in {DATA_FILE}: {sorted(missing_columns)}")

df[SOURCE_COL] = df[SOURCE_COL].map(clean_text)
if TARGET_COL in df.columns:
    df[TARGET_COL] = df[TARGET_COL].map(clean_text)

for column in PREDICTION_COLUMNS:
    if column not in df.columns:
        df[column] = pd.NA

if MAX_ROWS is not None:
    df = df.head(MAX_ROWS).copy()

completed_mask = df["levanti_harakat_prediction"].notna() & (df["levanti_harakat_prediction"].astype(str).str.len() > 0)
print("Rows:", len(df))
print("Already completed:", int(completed_mask.sum()))
print("Remaining:", int((~completed_mask).sum()))

Starting from the cleaned data file.
Rows: 36302
Already completed: 0
Remaining: 36302


In [14]:
####################################
# run predictions with periodic saving
####################################
# This loop is intentionally resumable. If Colab disconnects, rerun the cells
# above and this cell will skip rows that already have saved predictions.
rows_since_save = 0
pending_indices = df.index[~completed_mask].tolist()
errors = []

for row_idx in tqdm(pending_indices, desc="Predicting Levanti harakat"):
    source_text = clean_text(df.at[row_idx, SOURCE_COL])
    try:
        prediction = add_levanti_harakat(source_text)
    except Exception as exc:
        prediction = ""
        errors.append({"row_idx": row_idx, "source": source_text, "error": repr(exc)})

    df.at[row_idx, "levanti_harakat_prediction"] = prediction
    df.at[row_idx, "levanti_harakat_prediction_done_at"] = now_iso()

    rows_since_save += 1
    if rows_since_save >= SAVE_EVERY_ROWS:
        df.to_csv(PREDICTIONS_FILE, index=False)
        print(f"Saved progress through row {row_idx} to {PREDICTIONS_FILE}")
        rows_since_save = 0

df.to_csv(PREDICTIONS_FILE, index=False)
print("Prediction pass complete.")
print("Saved predictions to:", PREDICTIONS_FILE)
print("Rows with prediction errors:", len(errors))
if errors:
    display(pd.DataFrame(errors).head(20))

Predicting Levanti harakat:   0%|          | 0/36302 [00:00<?, ?it/s]

Saved progress through row 99 to /content/drive/MyDrive/arabizi/maknuune-v1.0.1_cleaned_levanti_harakat_predictions.csv
Saved progress through row 199 to /content/drive/MyDrive/arabizi/maknuune-v1.0.1_cleaned_levanti_harakat_predictions.csv
Saved progress through row 299 to /content/drive/MyDrive/arabizi/maknuune-v1.0.1_cleaned_levanti_harakat_predictions.csv
Saved progress through row 399 to /content/drive/MyDrive/arabizi/maknuune-v1.0.1_cleaned_levanti_harakat_predictions.csv
Saved progress through row 499 to /content/drive/MyDrive/arabizi/maknuune-v1.0.1_cleaned_levanti_harakat_predictions.csv
Saved progress through row 599 to /content/drive/MyDrive/arabizi/maknuune-v1.0.1_cleaned_levanti_harakat_predictions.csv
Saved progress through row 699 to /content/drive/MyDrive/arabizi/maknuune-v1.0.1_cleaned_levanti_harakat_predictions.csv
Saved progress through row 799 to /content/drive/MyDrive/arabizi/maknuune-v1.0.1_cleaned_levanti_harakat_predictions.csv
Saved progress through row 899 to

In [15]:
####################################
# exact-match accuracy metrics against gold harakat, if available
####################################
if TARGET_COL not in df.columns:
    print(f"Skipping metrics because {TARGET_COL!r} is not in the dataframe.")
else:
    completed_for_metrics = df["levanti_harakat_prediction"].notna() & (df["levanti_harakat_prediction"].astype(str).str.len() > 0)
    metrics_df = df.loc[completed_for_metrics].copy()
    if metrics_df.empty:
        raise ValueError("No completed predictions found yet. Run the prediction cell before calculating metrics.")

    metrics_df["arabic_stripped_length"] = metrics_df[SOURCE_COL].map(len)
    metrics_df["length_bucket"] = (metrics_df["arabic_stripped_length"] // 10 * 10).map(lambda start: f"{start}-{start + 9}")
    metrics_df["levanti_exact_match"] = metrics_df["levanti_harakat_prediction"].map(clean_text) == metrics_df[TARGET_COL].map(clean_text)

    overall_metrics = pd.DataFrame([
        {
            "group": "overall",
            "count": len(metrics_df),
            "levanti_exact_accuracy": metrics_df["levanti_exact_match"].mean(),
            "avg_source_length": metrics_df["arabic_stripped_length"].mean(),
        }
    ])

    length_metrics = (
        metrics_df.groupby("length_bucket", sort=False)
        .agg(
            count=(SOURCE_COL, "size"),
            levanti_exact_accuracy=("levanti_exact_match", "mean"),
            avg_source_length=("arabic_stripped_length", "mean"),
        )
        .reset_index()
        .rename(columns={"length_bucket": "group"})
    )

    all_metrics = pd.concat([overall_metrics, length_metrics], ignore_index=True)
    all_metrics.to_csv(METRICS_FILE, index=False)

    display(overall_metrics)
    display(length_metrics.sort_values("group"))
    print("Saved metrics to:", METRICS_FILE)

,group,count,levanti_exact_accuracy,avg_source_length
0,overall,36302,0.283676,5.066801


,group,count,levanti_exact_accuracy,avg_source_length
0,0-9,34072,0.299953,4.421196
1,10-19,1888,0.040784,12.676377
2,20-29,255,0.003922,23.556863
3,30-39,64,0.000000,33.750000
4,40-49,11,0.000000,45.545455
5,50-59,8,0.000000,51.625000
6,60-69,2,0.000000,66.000000
7,70-79,2,0.000000,75.000000


Saved metrics to: /content/drive/MyDrive/arabizi/levanti_harakat_prediction_metrics_by_length.csv


In [16]:
####################################
# preview saved predictions
####################################
preview_columns = [SOURCE_COL, "levanti_harakat_prediction"]
if TARGET_COL in df.columns:
    preview_columns.insert(1, TARGET_COL)
display(df[preview_columns].head(10))

,arabic_stripped,arabic_harakat,levanti_harakat_prediction
0,أبد,أَبَد,أَبَد
1,إبرة,إِبْرِة,إِبْرَة
2,إبر,إِبَر,إِبِر
3,قد خرم الإبرة,قَدّ خُرُم الإِبْرِة,قَد خَرَّم الإِبْرَة
4,إبرة العجوزة,إِبْرِة العَجُوزِة,إِبْرَة الْعَجُوزَِة
5,الإبرة غلبت الحايك,الإِبْرِة غلبت الحَايِك,الإِبْرَة غَلَّبِت الحَايِك
6,من إبرته,مِن إِبْرِتُه,مِن إِبْرِتُه
7,أباط,أَبَاط,أَبَّاط
8,باط,بَاط,بَاط
9,حطه تحت باطه,حَطُّه تحت بَاطُه,حَطُّه تَحِت بَاطُه
